# 02 Sentiment Analysis

Visualize Claude Haiku sentiment pipeline outputs (Gold layer).

This notebook:
1. Loads sentiment Gold layer via DuckDB
2. Shows overall sentiment distribution
3. Overlays sentiment factors on a K-line chart for a selected symbol
4. Highlights top positive and negative events

In [1]:
from pathlib import Path
import os

repo_root = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())

DATA_ROOT   = repo_root / 'data'
GOLD_GLOB   = str(DATA_ROOT / 'sentiment' / 'gold'   / '**' / '*.parquet')
SILVER_GLOB = str(DATA_ROOT / 'sentiment' / 'silver' / '**' / '*.parquet')
KLINE_GLOB  = str(DATA_ROOT / 'kline' / '**' / '*.parquet')

print(f'Repo root  : {repo_root}')
print(f'Gold glob  : {GOLD_GLOB}')
print(f'Silver glob: {SILVER_GLOB}')

Repo root  : /data00/home/guohuanwei.cztj/git_files/trade
Gold glob  : /data00/home/guohuanwei.cztj/git_files/trade/data/sentiment/gold/**/*.parquet
Silver glob: /data00/home/guohuanwei.cztj/git_files/trade/data/sentiment/silver/**/*.parquet


In [2]:
import duckdb
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

con = duckdb.connect()
print('DuckDB version:', duckdb.__version__)

DuckDB version: 1.4.4


## 1. Load Gold Layer

In [3]:
try:
    gold_df = con.execute(f"""
        SELECT *
        FROM read_parquet('{GOLD_GLOB}', union_by_name=true)
        ORDER BY date DESC, symbol
    """).df()
    print(f'Gold rows: {len(gold_df)}')
    print(f'Date range: {gold_df["date"].min()} to {gold_df["date"].max()}')
    print(f'Unique symbols: {gold_df["symbol"].nunique()}')
    gold_df.head(10)
except Exception as e:
    print(f'Gold layer not found: {e}')
    print('Run: python -m scripts.run_sentiment --date YYYY-MM-DD')
    gold_df = pd.DataFrame()

Gold rows: 216
Date range: 2026-03-02 to 2026-03-05
Unique symbols: 197


## 2. Sentiment Distribution

In [4]:
if not gold_df.empty:
    # Daily average sentiment
    daily_sent = gold_df.groupby('date').agg(
        mean_score=('sentiment_score', 'mean'),
        mean_net=('net_sentiment', 'mean'),
        total_articles=('article_count', 'sum'),
        symbol_count=('symbol', 'count'),
    ).reset_index().sort_values('date')

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=daily_sent['date'],
        y=daily_sent['mean_score'],
        marker_color=daily_sent['mean_score'].apply(
            lambda v: '#e84545' if v > 0 else '#2ba02b'
        ),
        name='Mean Sentiment Score',
    ))
    fig.add_hline(y=0, line_dash='dash', line_color='gray')
    fig.update_layout(
        title='Daily Market Sentiment Score (Claude Haiku)',
        xaxis_title='Date',
        yaxis_title='Avg Sentiment Score [-1, 1]',
        height=350,
    )
    fig.show()
else:
    print('No gold data. Run sentiment pipeline first.')

In [5]:
if not gold_df.empty:
    # Sentiment score histogram
    fig = px.histogram(
        gold_df,
        x='sentiment_score',
        nbins=40,
        title='Sentiment Score Distribution Across All Symbols',
        labels={'sentiment_score': 'Sentiment Score'},
        color_discrete_sequence=['steelblue'],
    )
    fig.add_vline(x=0, line_dash='dash', line_color='red')
    fig.update_layout(height=350)
    fig.show()

    # Net sentiment boxplot by date
    fig2 = px.box(
        gold_df,
        x='date',
        y='net_sentiment',
        title='Net Sentiment Distribution by Date',
        labels={'net_sentiment': 'Net Sentiment'},
    )
    fig2.update_layout(height=350)
    fig2.show()

## 3. K-line + Sentiment Overlay for a Symbol

In [6]:
# Select symbol: first from gold layer if available
if not gold_df.empty:
    sym_counts = gold_df.groupby('symbol').size().sort_values(ascending=False)
    SYMBOL = sym_counts.index[0] if not sym_counts.empty else '600000.SH'
else:
    SYMBOL = '600000.SH'

print(f'Plotting overlay for: {SYMBOL}')

Plotting overlay for: 000061.SZ


In [7]:
# Load kline data for the symbol
kline_df = pd.DataFrame()
try:
    kline_df = con.execute(f"""
        SELECT
            CAST(date AS VARCHAR) AS date,
            open, high, low, close, volume, prev_close
        FROM read_parquet('{KLINE_GLOB}', union_by_name=true)
        WHERE symbol = '{SYMBOL}'
        ORDER BY date
    """).df()
    print(f'Kline bars: {len(kline_df)}')
except Exception as e:
    print(f'Kline not available: {e}')

# Load gold sentiment for the symbol
sym_gold = pd.DataFrame()
if not gold_df.empty:
    sym_gold = gold_df[gold_df['symbol'] == SYMBOL].copy()
    sym_gold = sym_gold.sort_values('date')
    print(f'Sentiment rows: {len(sym_gold)}')

Kline bars: 0
Sentiment rows: 4


In [8]:
if not kline_df.empty or not sym_gold.empty:
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        row_heights=[0.55, 0.25, 0.20],
        vertical_spacing=0.05,
        subplot_titles=[f'{SYMBOL} K-line', 'Sentiment Score', 'Article Count'],
    )

    # Row 1: Candlestick
    if not kline_df.empty:
        fig.add_trace(go.Candlestick(
            x=kline_df['date'],
            open=kline_df['open'],
            high=kline_df['high'],
            low=kline_df['low'],
            close=kline_df['close'],
            name='Price',
            increasing_line_color='#e84545',
            decreasing_line_color='#2ba02b',
        ), row=1, col=1)

    # Row 2: Sentiment score
    if not sym_gold.empty:
        fig.add_trace(go.Bar(
            x=sym_gold['date'],
            y=sym_gold['sentiment_score'],
            marker_color=sym_gold['sentiment_score'].apply(
                lambda v: '#e84545' if v > 0.1 else ('#2ba02b' if v < -0.1 else '#aaaaaa')
            ),
            name='Sentiment',
        ), row=2, col=1)
        fig.add_hline(y=0, line_dash='dash', line_color='gray', row=2, col=1)

        # Row 3: Article count
        fig.add_trace(go.Bar(
            x=sym_gold['date'],
            y=sym_gold['article_count'],
            marker_color='steelblue',
            name='Articles',
        ), row=3, col=1)

    fig.update_layout(
        title=f'{SYMBOL} - Price & Sentiment Overlay',
        height=650,
        xaxis_rangeslider_visible=False,
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
    )
    fig.show()
else:
    print('No data available for overlay chart.')

## 4. Top Positive and Negative Events

In [9]:
# Load Silver layer for event details
silver_df = pd.DataFrame()
try:
    silver_df = con.execute(f"""
        SELECT date, symbol, title, sentiment_score, sentiment_label,
               event_type, event_magnitude, summary, confidence
        FROM read_parquet('{SILVER_GLOB}', union_by_name=true)
        ORDER BY ABS(sentiment_score) DESC
    """).df()
    print(f'Silver rows: {len(silver_df)}')
except Exception as e:
    print(f'Silver layer not available: {e}')

Silver rows: 684


In [10]:
if not silver_df.empty:
    print('=== TOP 10 POSITIVE EVENTS ===')
    top_pos = silver_df[silver_df['sentiment_label'] == 'positive'].nlargest(10, 'sentiment_score')
    display_cols = ['date', 'symbol', 'sentiment_score', 'event_type', 'event_magnitude', 'summary']
    print(top_pos[display_cols].to_string(index=False))

    print('\n=== TOP 10 NEGATIVE EVENTS ===')
    top_neg = silver_df[silver_df['sentiment_label'] == 'negative'].nsmallest(10, 'sentiment_score')
    print(top_neg[display_cols].to_string(index=False))
else:
    print('No silver data. Run sentiment pipeline first.')

=== TOP 10 POSITIVE EVENTS ===
      date    symbol  sentiment_score event_type  event_magnitude       summary
2026-03-05 301362.SZ              1.0  expansion              0.8   民爆光电股价创历史新高
2026-03-05  _MARKET_              1.0      other              0.8   佳鑫国际资源股价创新高
2026-03-05 300812.SZ              1.0      other              1.0   易天股份股价创历史新高
2026-03-05 002796.SZ              1.0  expansion              0.8     世嘉科技股价创新高
2026-03-05 300277.SZ              1.0  expansion              1.0      海联讯股价创新高
2026-03-05 002281.SZ              1.0    product              0.8   光迅科技股价创历史新高
2026-03-05 603269.SH              1.0  expansion              1.0   海鸥股份股价创历史新高
2026-03-05  _MARKET_              1.0  expansion              1.0 鸿承环保科技股价创历史新高
2026-03-05  _MARKET_              1.0  expansion              0.8   力量发展股价创历史新高
2026-03-05 603057.SH              1.0      other              0.8   紫燕食品股价创历史新高

=== TOP 10 NEGATIVE EVENTS ===
      date    symbol  sentiment_score event_type  event_m

In [11]:
if not silver_df.empty:
    # Event type breakdown
    event_counts = silver_df['event_type'].value_counts().reset_index()
    event_counts.columns = ['event_type', 'count']

    fig = px.pie(
        event_counts,
        names='event_type',
        values='count',
        title='News Event Type Distribution',
    )
    fig.update_layout(height=400)
    fig.show()

    # Sentiment by event type
    fig2 = px.box(
        silver_df,
        x='event_type',
        y='sentiment_score',
        title='Sentiment Score by Event Type',
        color='event_type',
    )
    fig2.add_hline(y=0, line_dash='dash', line_color='gray')
    fig2.update_layout(height=400, showlegend=False)
    fig2.show()

## 5. Sentiment Factor Correlation Analysis

In [12]:
if not gold_df.empty:
    factor_cols = ['net_sentiment', 'sentiment_score', 'sent_velocity', 'neg_shock', 'event_magnitude']
    available = [c for c in factor_cols if c in gold_df.columns]
    if available:
        corr = gold_df[available].corr()
        fig = px.imshow(
            corr,
            title='Sentiment Factor Correlation Matrix',
            color_continuous_scale='RdBu',
            zmin=-1, zmax=1,
            text_auto='.2f',
        )
        fig.update_layout(height=400)
        fig.show()
    else:
        print('Factor columns not found in gold data.')

con.close()